# Olist 쇼핑몰 주문 데이터 분석
어떤 고객에게 먼저 투자해야 할까?

- 데이터: Olist 공개 주문 데이터 (2016.09 ~ 2018.10), 분석 대상은 취소를 뺀 배송 완료 주문
- 도구: SQL(SQLite), Python(pandas, scipy)
- 먼저 `python run_analysis.py`를 실행하면 `output/`에 DB와 결과 파일이 생김

In [1]:
import sqlite3, pandas as pd
from pathlib import Path
con = sqlite3.connect('output/olist.db')
sql = lambda f: Path('sql', f).read_text(encoding='utf-8')
con.executescript(sql('00_base_views.sql'))
run = lambda f: pd.read_sql_query(sql(f), con)
pd.read_csv('output/table_sizes.csv', index_col=0)  # 사용한 표와 건수

,rows
customers,99441
orders,99441
order_items,112650
order_payments,103886
order_reviews,99224
products,32951
sellers,3095
category_translation,71


## 1. 배경과 문제
2018년부터 월 주문이 6천 ~ 7천 건에서 늘지 않음. 고객 대부분은 한 번만 구매

In [2]:
print(sql('06_monthly.sql'))
run('06_monthly.sql')

-- 월별 주문 수 (취소 제외)
SELECT strftime('%Y-%m', o.order_purchase_timestamp) AS month,
       COUNT(DISTINCT o.order_id) AS orders
FROM orders o
WHERE o.order_status <> 'canceled'
  AND o.order_purchase_timestamp >= '2017-01-01' AND o.order_purchase_timestamp < '2018-09-01'
GROUP BY month ORDER BY month;



,month,orders
0,2017-01,797
1,2017-02,1763
2,2017-03,2649
3,2017-04,2386
4,2017-05,3671
5,2017-06,3229
6,2017-07,3998
7,2017-08,4304
8,2017-09,4265
9,2017-10,4605


## 2. 고객 그룹 나누기 (RFM)
- R: 마지막 구매 후 일수 / F: 구매한 날 수 / M: 총 구매금액
- 그룹 1 재구매, 2 최근 우수, 3 최근 일반, 4 휴면 우수, 5 휴면 일반
- 우수 = 총 구매금액 R$110 이상(상위 40%), 휴면 = 마지막 구매 후 6개월(177일) 넘게 구매 없음
- 같은 날 추가 주문은 한 번으로 셈 (포함하면 재구매 3.0%, 제외하면 2.2%)

In [3]:
print(sql('01_rfm_groups.sql'))

-- RFM 고객 그룹: 최근성(R), 구매 횟수(F), 구매 금액(M)
-- R, M은 고객 전체를 5등분한 점수(5점이 가장 좋음). F는 서로 다른 날 2번 이상 구매하면 재구매 고객.
WITH snap AS (
  SELECT DATE(MAX(purchase_ts), '+1 day') AS d FROM v_orders
),
base AS (
  SELECT customer_unique_id,
         CAST(julianday((SELECT d FROM snap)) - julianday(MAX(purchase_ts)) AS INT) AS recency,
         COUNT(DISTINCT DATE(purchase_ts)) AS frequency,  -- 같은 날 추가 주문은 한 번으로 셈
         ROUND(SUM(order_value),2) AS monetary
  FROM v_orders
  GROUP BY customer_unique_id
),
scored AS (
  SELECT *,
         6 - NTILE(5) OVER (ORDER BY recency) AS r_score,
         NTILE(5) OVER (ORDER BY monetary)    AS m_score
  FROM base
)
SELECT *,
  CASE
    WHEN frequency >= 2                  THEN 1   -- 재구매 고객
    WHEN r_score >= 4 AND m_score >= 4   THEN 2   -- 최근 고액
    WHEN r_score >= 4                    THEN 3   -- 최근 소액
    WHEN m_score >= 4                    THEN 4   -- 이탈 고액
    ELSE 5                                        -- 이탈 소액
  END AS grp
FROM scored;



In [4]:
rfm = run('01_rfm_groups.sql')
names = {1: '재구매', 2: '최근 우수', 3: '최근 일반', 4: '휴면 우수', 5: '휴면 일반'}
g = rfm.groupby('grp').agg(customers=('monetary','size'), sales=('monetary','sum'), avg_amount=('monetary','mean'), avg_days=('recency','mean')).round(0)
g['name'] = g.index.map(names)
g['customer_pct'] = (g.customers / g.customers.sum() * 100).round(1)
g['sales_pct'] = (g.sales / g.sales.sum() * 100).round(1)
g

,customers,sales,avg_amount,avg_days,name,customer_pct,sales_pct
grp,,,,,,,
1,2015,546637.0,271.0,191.0,재구매,2.2,4.1
2,14562,3893772.0,267.0,91.0,최근 우수,15.6,29.5
3,21717,1205443.0,56.0,89.0,최근 일반,23.3,9.1
4,21198,5686113.0,268.0,335.0,휴면 우수,22.7,43.0
5,33858,1888284.0,56.0,337.0,휴면 일반,36.3,14.3


휴면 우수(4)와 최근 우수(2)의 평점, 늦은 주문 비율이 비슷해서 불만 때문에 떠난 것으로 보이지 않음

In [5]:
pd.read_csv('output/group2_vs_4.csv')

,grp,customers,review,late_pct
0,2,14659,4.15,8.24
1,4,21668,4.05,9.19


## 3. 어디에 집중할까
연락 1명당 기대 매출 = 평균 금액 x 3% (연락하면 3%가 다시 산다고 가정). 가정 비율이 바뀌어도 그룹 순위는 같음

In [6]:
g['per_contact'] = (g.avg_amount * 0.03).round(1)
g.loc[[4, 2, 1, 3, 5], ['name', 'customers', 'sales_pct', 'avg_amount', 'per_contact']]

,name,customers,sales_pct,avg_amount,per_contact
grp,,,,,
4,휴면 우수,21198,43.0,268.0,8.0
2,최근 우수,14562,29.5,267.0,8.0
1,재구매,2015,4.1,271.0,8.1
3,최근 일반,21717,9.1,56.0,1.7
5,휴면 일반,33858,14.3,56.0,1.7


### 휴면 우수 안에서 누구부터: 안 산 채로 시간이 지나면 돌아올 가능성
첫 구매 후 d일까지 안 산 고객이 다음 90일 안에 다시 산 비율. 12개월이 지나면 낮아짐

In [7]:
print(sql('08_repurchase_gap.sql'))
pd.read_csv('output/return_hazard.csv')

-- 재구매 고객이 첫 구매 후 며칠 만에 다시 샀는지 (같은 날 추가 주문은 제외)
WITH days AS (
  SELECT customer_unique_id, DATE(purchase_ts) AS d
  FROM v_orders GROUP BY customer_unique_id, DATE(purchase_ts)
),
ranked AS (
  SELECT customer_unique_id, d,
         ROW_NUMBER() OVER (PARTITION BY customer_unique_id ORDER BY d) AS rn
  FROM days
)
SELECT a.customer_unique_id,
       CAST(julianday(b.d) - julianday(a.d) AS INT) AS gap_days
FROM ranked a
JOIN ranked b ON a.customer_unique_id = b.customer_unique_id AND a.rn = 1 AND b.rn = 2;



,from_day,at_risk,returned,rate_pct
0,0,10818,140,1.29
1,90,10678,72,0.67
2,180,10606,75,0.71
3,270,10531,61,0.58
4,365,10470,39,0.37


In [8]:
pd.read_csv('output/targets.csv')  # 휴면 우수 전체, 마지막 구매 6~12개월

,dormant_all,target_6_12m,over_12m
0,21198,14237,6961


## 4. 제안의 근거: 무엇을 권할까
다시 살 때 같은 상품군을 사는 비율, 바우처 사용 여부별 재구매 비율

In [9]:
print(sql('09_order_main_category.sql'))
display(pd.read_csv('output/same_category.csv'))
pd.read_csv('output/voucher_repurchase.csv')

-- 주문별 대표 상품군 (주문에서 가장 비싼 상품의 상품군)
SELECT order_id, category
FROM (
  SELECT i.order_id, t.product_category_name_english AS category,
         ROW_NUMBER() OVER (PARTITION BY i.order_id ORDER BY i.price DESC) AS r
  FROM order_items i
  JOIN products p ON p.product_id = i.product_id
  JOIN category_translation t ON t.product_category_name = p.product_category_name
)
WHERE r = 1;



,same_category_pct,chance_pct,n
0,38.9,5.8,1970


,voucher,mean,sum,size,rate_pct
0,0.0,0.021302,1914,89849,2.13
1,1.0,0.028857,101,3500,2.89


## 5. 기대 효과 (가정)
연락으로 더 돌아오는 비율을 1, 3, 5%로 가정, 쿠폰 비용은 뺀 금액

In [10]:
pd.read_csv('output/scenario.csv')

,rate_pct,customers,extra_sales,vs_total_pct
0,1,142,38090.0,0.3
1,3,427,114538.0,0.9
2,5,712,190986.0,1.4


## 6. 한계와 더 나아갈 점
**한계**: 2년치 자료, 떠난 이유는 데이터에 없음, 연락 효과(3%)는 가정, 상품군·바우처 결과는 연관성, 우수 기준(R$110)은 직접 정한 값

**더 나아갈 점**: A/B 테스트로 효과 측정, 연락 비용 대비 이익 계산, 설문 등으로 떠난 이유 확인